In [2]:
import torch
from pathlib import Path
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-31B-it"
MODEL_DIR = Path("/workspaces/tengoku/models")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=MODEL_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    device_map="auto",
    cache_dir=MODEL_DIR,
    attn_implementation="sdpa",
)

# Optimize the model graph for your GPU — first inference will be slow (compile warmup),
# subsequent calls will be significantly faster
model = torch.compile(model, mode="reduce-overhead")

print(f"Loaded {MODEL_ID}")
print(f"Cached to: {MODEL_DIR}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

/workspaces/tengoku/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]/workspaces/tengoku/.venv/lib64/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 1188/1188 [00:07<00:00, 167.78it/s]


Loaded google/gemma-4-31B-it
Cached to: /workspaces/tengoku/models
Memory footprint: 17.79 GB


In [3]:
import threading
from transformers import TextIteratorStreamer
from IPython.display import display, Markdown, update_display

def chat(prompt: str, max_new_tokens: int = 1024, thinking: bool = False) -> None:
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt},
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking,
    )
    inputs = processor(text=text, return_tensors="pt").to(model.device)

    streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

    thread = threading.Thread(
        target=model.generate,
        kwargs={**inputs, "max_new_tokens": max_new_tokens, "streamer": streamer},
    )
    thread.start()

    display_handle = display(Markdown(""), display_id=True)
    buffer = ""
    for token in streamer:
        buffer += token
        update_display(Markdown(buffer), display_id=display_handle.display_id)

    thread.join()

In [4]:
chat("Tell me about the japanese word tengoku")

**Tengoku** (天国) is the Japanese word for **"Heaven"** or **"Paradise."**

To understand the word fully, it helps to look at its kanji (characters), its cultural context, and how it differs from other Japanese concepts of the afterlife.

### 1. Etymology (The Kanji)
The word is composed of two characters:
*   **天 (Ten):** This means "Heaven," "Sky," or "Imperial." It refers to the celestial realm or the overarching force of the universe.
*   **国 (Koku/Kuni):** This means "Country," "Land," or "Nation."

Literally, *Tengoku* translates to **"The Heavenly Country"** or **"The Land of Heaven."**

### 2. Religious Context
Unlike some words in Japanese that have ancient, indigenous roots, *Tengoku* is heavily influenced by foreign religions, specifically **Buddhism** and **Christianity**.

*   **Christianity:** *Tengoku* is the standard term used in Japanese Bibles and by Christians to describe the dwelling place of God and the destination for those who are saved.
*   **Buddhism:** While Buddhism has a complex system of "Pure Lands" (Jōdo) and various heavens (Deva realms), *Tengoku* is often used as a general, accessible term for a blissful afterlife where one is free from suffering.

### 3. Tengoku vs. Other Afterlife Terms
Japanese culture has several ways of describing "the other side," and they carry different nuances:

*   **Jōdo (浄土):** Meaning "Pure Land." This is a specifically Buddhist term. It is less about a "reward" for being good and more about a spiritual realm where one can achieve enlightenment more easily.
*   **Ano yo (あの世):** Literally "that world." This is a colloquial term for the afterlife in general (the "world beyond"). It is neutral and can refer to either heaven or hell.
*   **Meido (冥途):** This refers to the "dark path" or the underworld, often associated with the journey the soul takes after death.

### 4. Common Usage in Modern Culture
In modern Japanese, *Tengoku* is used both literally and metaphorically:

*   **Literally:** When talking about death or religion (e.g., *"He went to heaven"* $\rightarrow$ *Tengoku ni ikimashita*).
*   **Metaphorically:** To describe a place of extreme happiness or luxury on Earth. For example, if someone visits a beautiful tropical beach, they might say, *"Koko wa tengoku da!"* (**"This place is paradise!"**).

### Summary Table
| Term | Kanji | Meaning | Nuance |
| :--- | :--- | :--- | :--- |
| **Tengoku** | 天国 | Heaven/Paradise | Blissful, celestial, often religious or metaphorical. |
| **Jōdo** | 浄土 | Pure Land | Specifically Buddhist; a place for spiritual practice. |
| **Ano yo** | あの世 | The other world | General term for the afterlife. |

/workspaces/tengoku/.venv/lib64/python3.14/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
